# SAMURAI Optimize — VS Code Notebook (2-venv workflow)

Notebook để so sánh SAMURAI **gốc** vs **optimized** trên LaSOT-small.

## Workflow

Mỗi bản `sam2` có **venv riêng** → không conflict distribution, **không cần restart kernel** khi chuyển bản. Chỉ cần **Select Kernel** chọn đúng venv.

```
samurai_optimized/
├── .venv/              ← venv cho bản OPTIMIZED
├── sam2/               ← fork optimized (pip install -e vào .venv)
├── scripts/
├── samurai/
│   ├── .venv/          ← venv cho bản GỐC
│   ├── sam2/           ← SAMURAI gốc (pip install -e vào samurai/.venv)
│   └── scripts/
```

**Quy tắc duy nhất:** chọn đúng kernel trước khi chạy section A hoặc B.
- Chạy bản gốc → kernel `samurai/.venv/bin/python`
- Chạy bản optimized → kernel `.venv/bin/python`

Đọc `docs/2026-04-21-baseline-vs-optimized-runbook.md` để hiểu chi tiết vấn đề conflict.

## 0. Cấu hình đường dẫn (sửa cho phù hợp máy bạn)

In [ ]:
import os

# Sửa các path dưới đây nếu cần
REPO_ROOT   = "/home/ubuntu-phuocbh/Downloads/Khoa_luan_tot_nghiep_sam2/samurai_optimized"
DATA_ROOT   = "/path/to/lasot-small/small_LaSOT"           # <-- SỬA: thư mục chứa LaSOT
TESTING_SET = os.path.join(REPO_ROOT, "testing_set.txt")    # ghi ngay trong repo cho tiện

os.environ["REPO_ROOT"]   = REPO_ROOT
os.environ["DATA_ROOT"]   = DATA_ROOT
os.environ["TESTING_SET"] = TESTING_SET

print("REPO_ROOT  =", REPO_ROOT)
print("DATA_ROOT  =", DATA_ROOT)
print("TESTING_SET=", TESTING_SET)

## 1. Tạo 2 venv (chạy 1 lần duy nhất)

Chạy trong terminal (không chạy trong notebook) hoặc chạy các cell dưới.

> Nếu đã tạo venv rồi thì **bỏ qua** section này.

### 1a. Venv cho bản OPTIMIZED (`samurai_optimized/.venv`)

In [ ]:
%%bash
set -e
cd "$REPO_ROOT"
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
pip install -e sam2/
pip install matplotlib==3.7 tikzplotlib jpeg4py opencv-python lmdb pandas scipy loguru psutil
echo "=== DONE: venv optimized ==="
which python
python -c "import sam2; print('sam2:', sam2.__file__)" 

### 1b. Venv cho bản GỐC (`samurai_optimized/samurai/.venv`)

In [ ]:
%%bash
set -e
cd "$REPO_ROOT/samurai"
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
pip install -e sam2/
pip install matplotlib==3.7 tikzplotlib jpeg4py opencv-python lmdb pandas scipy loguru psutil
echo "=== DONE: venv baseline ==="
which python
python -c "import sam2; print('sam2:', sam2.__file__)" 

### 1c. Đăng ký kernel cho VS Code

Sau khi tạo xong 2 venv, vào VS Code:
1. **Command Palette** (`Ctrl+Shift+P`) → `Python: Select Interpreter` → `Enter interpreter path...`
2. Trỏ lần lượt tới:
   - `samurai_optimized/.venv/bin/python`  (optimized)
   - `samurai_optimized/samurai/.venv/bin/python`  (baseline)
3. Sau đó khi mở notebook, click **Select Kernel** góc trên-phải sẽ thấy cả 2 venv.

## 2. Download checkpoints (chạy 1 lần)

In [ ]:
# Checkpoints cho bản optimized
!cd "$REPO_ROOT/sam2/checkpoints" && bash download_ckpts.sh

In [ ]:
# Checkpoints cho bản gốc
!cd "$REPO_ROOT/samurai/sam2/checkpoints" && bash download_ckpts.sh

## 3. Tạo testing set

In [ ]:
import os
os.makedirs(os.path.dirname(TESTING_SET) or ".", exist_ok=True)
with open(TESTING_SET, "w") as f:
    f.write("mouse-1\nelectricfan-1\ngecko-1\n")
print("Wrote", TESTING_SET)
print(open(TESTING_SET).read())

---

## A. Chạy bản SAMURAI **GỐC**

### Chọn kernel: `samurai/.venv/bin/python`

Click **Select Kernel** (góc trên-phải) → chọn interpreter `samurai_optimized/samurai/.venv/bin/python`.

Sau khi chọn xong, **chạy lại Cell 0** (cấu hình path) rồi tiếp tục các cell bên dưới.

In [ ]:
# A.1 — Verify đúng venv baseline
import sys, sam2, torch
print("Python    :", sys.executable)
print("sam2 file :", sam2.__file__)
print("CUDA      :", torch.cuda.is_available())

assert sam2.__file__ is not None, \
    "sam2.__file__ = None → namespace fallback. Chạy lại pip install -e trong venv baseline."
assert "samurai/sam2" in sam2.__file__.replace("\\", "/"), \
    f"Load nhầm bản: {sam2.__file__}. Phải chứa 'samurai/sam2'. Đã chọn đúng kernel chưa?"
print("\nOK — đang dùng bản GỐC")

### A.2 — Chạy inference baseline

In [ ]:
!python "$REPO_ROOT/samurai/scripts/main_inference.py" \
    --model_name base_plus \
    --evaluate \
    --data_root "$DATA_ROOT" \
    --testing_set "$TESTING_SET" 

---

## B. Chạy bản **OPTIMIZED**

### Chọn kernel: `.venv/bin/python`

Click **Select Kernel** (góc trên-phải) → chọn interpreter `samurai_optimized/.venv/bin/python`.

Sau khi chọn xong, **chạy lại Cell 0** (cấu hình path) rồi tiếp tục các cell bên dưới.

In [ ]:
# B.1 — Verify đúng venv optimized
import sys, sam2, torch
print("Python    :", sys.executable)
print("sam2 file :", sam2.__file__)
print("CUDA      :", torch.cuda.is_available())

assert sam2.__file__ is not None, \
    "sam2.__file__ = None → namespace fallback. Chạy lại pip install -e trong venv optimized."
p = sam2.__file__.replace("\\", "/")
assert "/samurai/" not in p and "samurai_optimized/sam2" in p, \
    f"Load nhầm bản: {sam2.__file__}. Phải trỏ tới samurai_optimized/sam2/. Đã chọn đúng kernel chưa?"
print("\nOK — đang dùng bản OPTIMIZED")

### B.2 — Chạy inference optimized

Flag `--optimized` **bắt buộc**, nếu không `release_interval=0` → bypass toàn bộ logic tối ưu.

In [ ]:
!python "$REPO_ROOT/scripts/main_inference.py" \
    --optimized \
    --model_name base_plus \
    --log_metrics \
    --run_tag optimized \
    --evaluate \
    --data_root "$DATA_ROOT" \
    --testing_set "$TESTING_SET" 

Các flag tối ưu có thể tinh chỉnh:

| Flag | Default | Ý nghĩa |
|---|---:|---|
| `--release_interval` | 60 | Mỗi N frame giải phóng frame cũ |
| `--keep_window_maskmem` | 1000 | Số frame giữ maskmem_features |
| `--keep_window_pred_masks` | 60 | Số frame giữ pred_masks |
| `--max_cache_frames` | 60 | LRU cache images trong RAM |
| `--no_auto_promote` | off | Tắt auto-promote cond frames |

---

## C. So sánh kết quả

Chạy cell này với **kernel nào cũng được** (chỉ cần matplotlib + pandas).

In [ ]:
!python "$REPO_ROOT/scripts/plot_metrics.py" \
    --run "$REPO_ROOT/metrics/samurai_base_plus/baseline" \
    --run "$REPO_ROOT/metrics/samurai_base_plus/optimized" \
    --label Baseline --label Optimized \
    --mode per_video